# ROI Analysis and UMAP Visualization Workflow

This notebook demonstrates how to:
1. **Define ROIs in napari** - Draw regions of interest on spatial data
2. **Extract cells within ROIs** - Get cells that fall within defined regions
3. **Perform UMAP clustering** - Cluster cells within each ROI
4. **Visualize results** - Create comparative plots and analysis

## Prerequisites
- Combined spatial data with cell type annotations
- napari for interactive ROI definition
- Standard single-cell analysis packages (scanpy, etc.)

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import spatialdata as sd
from roi_umap_analysis import ROIAnalyzer
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.rcParams['figure.figsize'] = (10, 6)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=80, facecolor='white')

## Step 1: Load Spatial Data

First, let's load our combined spatial data that includes cell type annotations.

In [ ]:
# Load the annotated spatial data
zarr_path = "../combined_direct_coords_annotated.zarr"
analyzer = ROIAnalyzer(zarr_path)

print(f"Loaded spatial data:")
print(f"  - Cells: {analyzer.table.n_obs:,}")
print(f"  - Genes: {analyzer.table.n_vars:,}")
print(f"  - Images: {len(analyzer.sdata.images)}")
print(f"  - Shapes: {len(analyzer.sdata.shapes)}")
print(f"  - Points: {len(analyzer.sdata.points)}")

## Step 2: Define ROIs in napari

**Option A: Interactive ROI Definition**

Run the following cell to open napari and define ROIs interactively:

In [ ]:
# Option A: Define ROIs interactively in napari
# Uncomment to run:

# roi_data = analyzer.launch_napari_for_roi_definition(save_path="my_rois.json")
# print(f"Defined {len(roi_data)} ROIs")
# for name, info in roi_data.items():
#     print(f"  {name}: {info['area']:.0f} area units")

print("To define ROIs interactively:")
print("1. Uncomment the code above")
print("2. Run the cell")
print("3. napari will open with your spatial data")
print("4. Use the 'ROIs' shapes layer to draw polygons")
print("5. Close napari when done")

**Option B: Create Example ROIs**

For demonstration purposes, let's create some example ROIs:

In [ ]:
# Option B: Create example ROIs for demonstration
import json

# Create example ROIs that cover different areas of the combined tissue
example_rois = {
    "Sample_007_Region": {
        "coordinates": [
            [2000, 0],
            [6000, 0],
            [6000, 1500],
            [2000, 1500]
        ],
        "area": 6000000,
        "bounds": [2000, 0, 6000, 1500]
    },
    "Sample_117_Region": {
        "coordinates": [
            [2000, 3000],
            [6000, 3000],
            [6000, 4500],
            [2000, 4500]
        ],
        "area": 6000000,
        "bounds": [2000, 3000, 6000, 4500]
    },
    "Border_Region": {
        "coordinates": [
            [3000, 1800],
            [5000, 1800],
            [5000, 2700],
            [3000, 2700]
        ],
        "area": 1800000,
        "bounds": [3000, 1800, 5000, 2700]
    }
}

# Save example ROIs
with open("../example_rois.json", "w") as f:
    json.dump(example_rois, f, indent=2)

# Load the example ROIs
analyzer.load_roi_data("../example_rois.json")

print(f"Created {len(analyzer.roi_data)} example ROIs:")
for name, info in analyzer.roi_data.items():
    print(f"  {name}: {info['area']:,.0f} area units")

## Step 3: Extract Cells Within ROIs

Now let's find which cells fall within each ROI:

In [ ]:
# Extract cells within each ROI
roi_cells = analyzer.extract_cells_in_rois()

print("Cells found in each ROI:")
total_cells = 0
for roi_name, cell_indices in roi_cells.items():
    print(f"  {roi_name}: {len(cell_indices):,} cells")
    total_cells += len(cell_indices)

print(f"\nTotal cells in all ROIs: {total_cells:,}")
print(f"Percentage of total cells: {total_cells/analyzer.table.n_obs*100:.1f}%")

## Step 4: Perform UMAP Analysis

For each ROI, we'll perform:
- Data normalization and scaling
- Principal component analysis (PCA)
- UMAP dimensionality reduction
- Leiden clustering

In [ ]:
# Perform UMAP analysis on each ROI
results = analyzer.perform_umap_analysis(roi_cells, output_dir="../roi_analysis_results")

print("UMAP Analysis Results:")
for roi_name, roi_result in results.items():
    adata = roi_result['adata']
    n_cells = roi_result['n_cells']
    n_clusters = len(adata.obs['leiden'].unique())
    
    print(f"\n{roi_name}:")
    print(f"  - Cells analyzed: {n_cells:,}")
    print(f"  - Leiden clusters: {n_clusters}")
    print(f"  - Genes used: {adata.n_vars}")
    
    # Show cluster sizes
    cluster_sizes = adata.obs['leiden'].value_counts().sort_index()
    print(f"  - Cluster sizes: {dict(cluster_sizes)}")
    
    # Show top cell types if available
    if 'cell_type_predicted' in adata.obs.columns:
        top_types = adata.obs['cell_type_predicted'].value_counts().head(3)
        print(f"  - Top cell types: {dict(top_types)}")

## Step 5: Visualize Individual ROI Results

Let's look at the UMAP plots for each ROI:

In [ ]:
# Display UMAP plots for each ROI
for roi_name, roi_result in results.items():
    adata = roi_result['adata']
    
    print(f"\n{roi_name} - UMAP Analysis")
    print("=" * 50)
    
    # Create subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # UMAP by clusters
    sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, frameon=False, legend_loc='on data')
    axes[0].set_title(f'{roi_name} - Leiden Clusters')
    
    # UMAP by cell type
    if 'cell_type_predicted' in adata.obs.columns:
        sc.pl.umap(adata, color='cell_type_predicted', ax=axes[1], show=False, frameon=False)
        axes[1].set_title(f'{roi_name} - Cell Types')
    else:
        axes[1].text(0.5, 0.5, 'No cell type\nannotations', ha='center', va='center', 
                    transform=axes[1].transAxes, fontsize=14)
        axes[1].set_title(f'{roi_name} - Cell Types (N/A)')
    
    # UMAP by total counts
    sc.pl.umap(adata, color='total_counts', ax=axes[2], show=False, frameon=False)
    axes[2].set_title(f'{roi_name} - Total UMI Count')
    
    plt.tight_layout()
    plt.show()
    
    # Show cluster composition if cell types available
    if 'cell_type_predicted' in adata.obs.columns:
        print(f"\nCluster composition for {roi_name}:")
        comp_df = pd.crosstab(adata.obs['leiden'], adata.obs['cell_type_predicted'])
        print(comp_df)
        
        # Plot composition heatmap
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.heatmap(comp_df.T, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f'{roi_name} - Cell Type Counts by Cluster')
        ax.set_ylabel('Cell Type')
        ax.set_xlabel('Leiden Cluster')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## Step 6: Combined Analysis Across ROIs

Let's create a combined analysis to compare ROIs:

In [ ]:
# Create combined analysis across all ROIs
combined_df = analyzer.create_combined_analysis(results, output_dir="../roi_analysis_results")

print("Combined Analysis Summary:")
print(f"Total cells analyzed: {len(combined_df):,}")
print(f"ROIs analyzed: {combined_df['roi'].nunique()}")
print(f"Total clusters found: {combined_df['leiden_cluster'].nunique()}")

if 'cell_type' in combined_df.columns:
    print(f"Cell types identified: {combined_df['cell_type'].nunique()}")

# Display the first few rows
print("\nFirst 10 rows of combined data:")
combined_df.head(10)

## Step 7: Comparative Visualizations

Let's create some comparative plots across ROIs:

In [ ]:
# ROI comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Cell counts per ROI
roi_counts = combined_df['roi'].value_counts()
roi_counts.plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Number of Cells per ROI', fontsize=14)
axes[0, 0].set_ylabel('Cell Count')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Gene expression distribution by ROI
sns.boxplot(data=combined_df, x='roi', y='n_genes', ax=axes[0, 1])
axes[0, 1].set_title('Gene Expression Distribution by ROI', fontsize=14)
axes[0, 1].set_ylabel('Number of Genes')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Total UMI counts by ROI
sns.boxplot(data=combined_df, x='roi', y='total_counts', ax=axes[1, 0])
axes[1, 0].set_title('Total UMI Count Distribution by ROI', fontsize=14)
axes[1, 0].set_ylabel('Total UMI Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Cell type distribution by ROI
if 'cell_type' in combined_df.columns:
    celltype_counts = combined_df.groupby(['roi', 'cell_type']).size().unstack(fill_value=0)
    celltype_counts.plot(kind='bar', stacked=True, ax=axes[1, 1])
    axes[1, 1].set_title('Cell Type Distribution by ROI', fontsize=14)
    axes[1, 1].set_ylabel('Cell Count')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
else:
    axes[1, 1].text(0.5, 0.5, 'No cell type\nannotations available', 
                   ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=14)
    axes[1, 1].set_title('Cell Type Distribution (N/A)', fontsize=14)

plt.tight_layout()
plt.show()

## Step 8: UMAP Comparison Across ROIs

Let's create a combined UMAP plot showing all ROIs:

In [ ]:
# Create combined UMAP plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# UMAP colored by ROI
for roi_name in combined_df['roi'].unique():
    roi_data = combined_df[combined_df['roi'] == roi_name]
    axes[0].scatter(roi_data['umap_1'], roi_data['umap_2'], 
                   label=roi_name, alpha=0.6, s=10)

axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')
axes[0].set_title('UMAP Colored by ROI')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# UMAP colored by cluster
for cluster in combined_df['leiden_cluster'].unique():
    cluster_data = combined_df[combined_df['leiden_cluster'] == cluster]
    axes[1].scatter(cluster_data['umap_1'], cluster_data['umap_2'], 
                   label=f'Cluster {cluster}', alpha=0.6, s=10)

axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')
axes[1].set_title('UMAP Colored by Leiden Cluster')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9: Statistical Analysis

Let's perform some statistical comparisons between ROIs:

In [ ]:
# Statistical comparison between ROIs
from scipy import stats

print("Statistical Analysis Between ROIs")
print("=" * 40)

# Compare gene expression between ROIs
roi_names = combined_df['roi'].unique()
if len(roi_names) >= 2:
    roi1_data = combined_df[combined_df['roi'] == roi_names[0]]
    roi2_data = combined_df[combined_df['roi'] == roi_names[1]]
    
    # T-test for gene expression
    t_stat, p_val = stats.ttest_ind(roi1_data['n_genes'], roi2_data['n_genes'])
    print(f"\nGene Expression Comparison ({roi_names[0]} vs {roi_names[1]}):")
    print(f"  Mean genes {roi_names[0]}: {roi1_data['n_genes'].mean():.1f}")
    print(f"  Mean genes {roi_names[1]}: {roi2_data['n_genes'].mean():.1f}")
    print(f"  T-statistic: {t_stat:.3f}")
    print(f"  P-value: {p_val:.6f}")
    
    # T-test for UMI counts
    t_stat, p_val = stats.ttest_ind(roi1_data['total_counts'], roi2_data['total_counts'])
    print(f"\nUMI Count Comparison ({roi_names[0]} vs {roi_names[1]}):")
    print(f"  Mean UMI {roi_names[0]}: {roi1_data['total_counts'].mean():.1f}")
    print(f"  Mean UMI {roi_names[1]}: {roi2_data['total_counts'].mean():.1f}")
    print(f"  T-statistic: {t_stat:.3f}")
    print(f"  P-value: {p_val:.6f}")

# Summary statistics for each ROI
print("\nSummary Statistics by ROI:")
summary_stats = combined_df.groupby('roi').agg({
    'n_genes': ['mean', 'std', 'min', 'max'],
    'total_counts': ['mean', 'std', 'min', 'max'],
    'leiden_cluster': 'nunique'
}).round(2)

print(summary_stats)

## Step 10: Save Results

Finally, let's save our analysis results:

In [ ]:
# Save results
output_dir = "../roi_analysis_results"

# Save combined dataframe
combined_df.to_csv(f"{output_dir}/combined_roi_analysis.csv", index=False)

# Save individual ROI results
for roi_name, roi_result in results.items():
    adata = roi_result['adata']
    
    # Save as h5ad file
    adata.write(f"{output_dir}/{roi_name}_analysis.h5ad")
    
    # Save cluster assignments
    cluster_df = pd.DataFrame({
        'cell_id': adata.obs.index,
        'leiden_cluster': adata.obs['leiden'],
        'umap_1': adata.obsm['X_umap'][:, 0],
        'umap_2': adata.obsm['X_umap'][:, 1]
    })
    
    if 'cell_type_predicted' in adata.obs.columns:
        cluster_df['cell_type'] = adata.obs['cell_type_predicted']
    
    cluster_df.to_csv(f"{output_dir}/{roi_name}_clusters.csv", index=False)

print("Results saved to:")
print(f"  - Combined analysis: {output_dir}/combined_roi_analysis.csv")
print(f"  - Individual ROI analyses: {output_dir}/[ROI_NAME]_analysis.h5ad")
print(f"  - Cluster assignments: {output_dir}/[ROI_NAME]_clusters.csv")
print(f"  - Visualization plots: {output_dir}/[ROI_NAME]_*.png")

## Summary

This workflow demonstrated how to:

1. ✅ **Define ROIs** - Created regions of interest on spatial data
2. ✅ **Extract cells** - Found cells within each ROI
3. ✅ **Perform UMAP** - Clustered cells within each ROI
4. ✅ **Visualize results** - Created comparative plots and analysis
5. ✅ **Save outputs** - Exported results for further analysis

### Key Insights

- Each ROI can have distinct clustering patterns
- Cell type composition varies between regions
- Gene expression profiles differ across ROIs
- Statistical comparisons reveal region-specific differences

### Next Steps

- **Differential gene expression** analysis between ROIs
- **Pathway enrichment** analysis for each ROI
- **Spatial neighborhood** analysis within ROIs
- **Trajectory analysis** if time-series data available

### Files Generated

All analysis results are saved in the `roi_analysis_results/` directory for further exploration.